# Model Building

In this notebook, multiple machine learning algorithms are trained and compared to predict flight arrival delays.

The preprocessing pipeline developed in the previous notebook is integrated with each model to ensure consistent data transformations during training and evaluation.

The models are evaluated using the validation dataset (2022), while the test dataset (2023) is reserved for the final evaluation in the next notebook.

The following models will be implemented:

- Logistic Regression
- Random Forest
- XGBoost

Hyperparameter tuning will be performed to improve model performance, and the best-performing model will be selected for deployment.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    TargetEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from pathlib import Path


def find_project_root():
    """
    Locate the project root regardless of whether the notebook
    is launched from the project root or the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Keep the data and notebooks folders inside the same project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
selected_features = [

    "AIRLINE",

    "ORIGIN",
    "DEST",

    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "QUARTER",

    "DEP_HOUR",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"

]

target = "IS_DELAYED"

## Chronological Data Splitting

The dataset is divided chronologically:

- Training data: 2019–2021
- Validation data: 2022
- Test data: 2023

This approach preserves the temporal order of the data and provides a more realistic evaluation of future flight-delay predictions.

In [ ]:
train_df = df[df["YEAR"] <= 2021].copy()

validation_df = df[df["YEAR"] == 2022].copy()

test_df = df[df["YEAR"] == 2023].copy()

In [ ]:
X_train = train_df[selected_features]
y_train = train_df[target]

X_validation = validation_df[selected_features]
y_validation = validation_df[target]

X_test = test_df[selected_features]
y_test = test_df[target]

In [ ]:
split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Period": ["2019–2021", "2022", "2023"],
    "Rows": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "Delayed Flights": [
        y_train.sum(),
        y_validation.sum(),
        y_test.sum()
    ],
    "Delay Rate": [
        y_train.mean(),
        y_validation.mean(),
        y_test.mean()
    ]
})

split_summary

## Target Distribution

The target variable is imbalanced because on-time flights occur more frequently than delayed flights. Therefore, Accuracy is reported together with Precision, Recall, F1-score, and ROC-AUC.

In [ ]:
class_distribution = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Proportion": y_train.value_counts(normalize=True)
}).rename(index={
    0: "On Time",
    1: "Delayed"
})

class_distribution

In [ ]:
majority_baseline_accuracy = y_validation.value_counts(
    normalize=True
).max()

print(
    f"Majority-class baseline accuracy: "
    f"{majority_baseline_accuracy:.4f}"
)

A classifier that predicts every validation flight as on time would achieve approximately 79.68% accuracy. Therefore, model accuracy must be interpreted relative to this baseline and alongside Recall and F1-score.

In [ ]:
numerical_features = [

    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "QUARTER",

    "DEP_HOUR",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"

]

In [ ]:
one_hot_features = [

    "DAY_OF_WEEK",

    "TIME_OF_DAY",

    "SEASON",

    "DISTANCE_CATEGORY"

]

In [ ]:
target_encode_features = [

    "AIRLINE",

    "ORIGIN",

    "DEST"

]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [ ]:
one_hot_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
target_transformer = Pipeline(
    steps=[
        (
            "target_encoder",
            TargetEncoder(
                target_type="binary",
                random_state=42
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_transformer,
            numerical_features
        ),

        (
            "onehot",
            one_hot_transformer,
            one_hot_features
        ),

        (
            "target",
            target_transformer,
            target_encode_features
        )

    ],

    remainder="drop"

)

## Model Evaluation Function

To avoid repeating evaluation code for each machine learning model, a reusable function is created. This function computes the most common classification performance metrics and returns them in a structured format.

The evaluated metrics are:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC Score

These metrics will be used consistently for comparing all models.

In [ ]:
def evaluate_model(model, X, y, model_name="Model"):
    """
    Evaluate a classification model and display its performance metrics.
    """

    # Predictions
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    # Metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    roc_auc = roc_auc_score(y, y_prob)

    # Results table
    results = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ],
        "Score": [
            accuracy,
            precision,
            recall,
            f1,
            roc_auc
        ]
    })

    print(f"\n{'='*60}")
    print(f"{model_name} Performance")
    print(f"{'='*60}")

    display(results)

    print("\nClassification Report\n")
    print(classification_report(y, y_pred))

    ConfusionMatrixDisplay.from_predictions(y, y_pred)
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    }

## Logistic Regression

Logistic Regression is used as the baseline classification model. It provides a simple and interpretable benchmark against which more complex machine learning models can be compared.

The preprocessing steps and Logistic Regression classifier are combined into a single pipeline. This ensures that all transformations are learned only from the training data, which prevents data leakage.

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                
                random_state=42
            )
        )
    ]
)

logistic_pipeline

In [ ]:
print("Training Logistic Regression...")

logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression training completed.")

In [ ]:
logistic_results = evaluate_model(
    model=logistic_pipeline,
    X=X_validation,
    y=y_validation,
    model_name="Logistic Regression"
)

## Random Forest

Random Forest is an ensemble learning algorithm that combines multiple decision trees to capture nonlinear relationships and feature interactions.

Because delayed flights represent the minority class, balanced class weights are used to give greater importance to delayed observations during training.

The same preprocessing pipeline is applied to maintain consistency across all models.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
           RandomForestClassifier(
                n_estimators=100,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_pipeline

In [ ]:
print("Training Random Forest...")

rf_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

In [ ]:
rf_results = evaluate_model(
    model=rf_pipeline,
    X=X_validation,
    y=y_validation,
    model_name="Random Forest"
)

## XGBoost

Extreme Gradient Boosting (XGBoost) is an advanced ensemble learning algorithm based on gradient boosting decision trees.

XGBoost builds trees sequentially, where each new tree corrects the errors made by the previous trees. It incorporates regularization techniques to reduce overfitting and often achieves state-of-the-art performance on structured datasets.

The same preprocessing pipeline is used to ensure a fair comparison with the previous models.

In [ ]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_pipeline

In [ ]:
print("Training XGBoost...")

xgb_pipeline.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

In [ ]:
xgb_results = evaluate_model(
    model=xgb_pipeline,
    X=X_validation,
    y=y_validation,
    model_name="XGBoost"
)

In [ ]:
comparison = pd.DataFrame([
    logistic_results,
    rf_results,
    xgb_results
])

comparison.sort_values(
    by="ROC_AUC",
    ascending=False
).reset_index(drop=True)

The baseline models showed that the dataset suffers from class imbalance. Although Logistic Regression and XGBoost achieved higher accuracy, they identified very few delayed flights. Random Forest produced the highest Recall and F1-score but lower ROC-AUC. Therefore, further hyperparameter tuning and imbalance handling are required before selecting the final model.

## XGBoost Hyperparameter Tuning

XGBoost is selected for further tuning because it achieved the highest baseline validation accuracy while also producing probability estimates suitable for further optimization.

Due to the size of the training dataset, RandomizedSearchCV is applied to a stratified sample of 400,000 training observations. Multiple evaluation metrics are recorded, while Accuracy is used as the refitting criterion because of the project requirement.

In [ ]:
from sklearn.model_selection import train_test_split

X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=400_000,
    stratify=y_train,
    random_state=42
)

print("Tuning sample shape:", X_tune.shape)
print("\nClass proportions:")
print(y_tune.value_counts(normalize=True))

In [ ]:
xgb_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
param_distributions = {
    "classifier__n_estimators": [150, 250, 350, 500],
    "classifier__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "classifier__max_depth": [3, 4, 5, 6, 8],
    "classifier__min_child_weight": [1, 3, 5, 10],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__gamma": [0, 0.1, 0.3, 0.5],
    "classifier__reg_alpha": [0, 0.01, 0.1, 0.5],
    "classifier__reg_lambda": [1, 2, 5, 10]
}

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [ ]:
xgb_random_search = RandomizedSearchCV(
    estimator=xgb_tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=scoring,
    refit="accuracy",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1,
    return_train_score=False
)

In [ ]:
print("Starting XGBoost hyperparameter tuning...")

xgb_random_search.fit(X_tune, y_tune)

print("Hyperparameter tuning completed.")

In [ ]:
print("Best cross-validation accuracy:")
print(xgb_random_search.best_score_)

print("\nBest parameters:")
xgb_random_search.best_params_

In [ ]:
cv_results = pd.DataFrame(xgb_random_search.cv_results_)

result_columns = [
    "rank_test_accuracy",
    "mean_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "params"
]

cv_results[result_columns].sort_values(
    by="rank_test_accuracy"
).head(10)

In [ ]:
best_xgb_params = {
    key.replace("classifier__", ""): value
    for key, value in xgb_random_search.best_params_.items()
}

best_xgb_params

In [ ]:
tuned_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                **best_xgb_params,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
print("Training tuned XGBoost on the full training set...")

tuned_xgb_pipeline.fit(X_train, y_train)

print("Tuned XGBoost training completed.")

In [ ]:
tuned_xgb_results = evaluate_model(
    model=tuned_xgb_pipeline,
    X=X_validation,
    y=y_validation,
    model_name="Tuned XGBoost"
)

In [ ]:
comparison = pd.DataFrame([
    logistic_results,
    rf_results,
    xgb_results,
    tuned_xgb_results
])

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

comparison

In [ ]:
comparison["Accuracy_Above_Majority_Baseline"] = (
    comparison["Accuracy"] - majority_baseline_accuracy
)

comparison

In [ ]:
(y_validation == 0).mean()

## Model Building Conclusion

Logistic Regression, Random Forest, and XGBoost were evaluated using the 2022 validation dataset.

XGBoost achieved the highest validation accuracy, while Logistic Regression achieved the highest ROC-AUC by a small margin. Random Forest achieved the highest Recall and F1-score for detecting delayed flights.

Hyperparameter tuning was applied to XGBoost using RandomizedSearchCV. The tuned model slightly improved Recall and F1-score, but its Accuracy and ROC-AUC decreased compared with the baseline XGBoost model.

The baseline XGBoost model remained the highest-accuracy model. However, its accuracy was only slightly higher than the majority-class baseline, indicating that the existing pre-departure features contain limited predictive information.

Additional model-improvement experiments will be conducted in the next notebook. The 2023 test dataset remains untouched for final evaluation.